# Chapter 13 &mdash; The Chomsky Hierarchy: Machines, Languages, Grammars

**Concept 11 of the Chapter 13 decomposition:** *The Chomsky Hierarchy: Machines, Languages, and Grammars Unified*

DFA/regular, DPDA/DCFL, NPDA/CFL, LBA/CSL, TM/RE &mdash; each family strictly inside the next.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter13/Concept-Chomsky-Hierarchy/Concept-Chomsky-Hierarchy.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_PDA        import *
from jove.Def_TM         import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


Everything in the book, in one table:

| machine | languages | grammars | example |
|---|---|---|---|
| DFA = NFA | regular | right-linear | $(01)^*$ |
| DPDA | deterministic CFL | LR(k) | $w\#w^R$ |
| NPDA | context-free | context-free | $ww^R$, Dyck |
| LBA | context-sensitive | context-sensitive | $a^nb^nc^n$ |
| TM | recursively enumerable | unrestricted | $A_{TM}$ |

Every containment is **strict**, and you have already seen a witness for each:

* regular $\subsetneq$ DCFL: $\{a^nb^n\}$ (Chapter 4);
* DCFL $\subsetneq$ CFL: $\{ww^R\}$ (Chapter 11, Concept 17);
* CFL $\subsetneq$ CSL: $\{a^nb^nc^n\}$ (Chapter 11, Concept 19);
* CSL $\subsetneq$ RE: $A_{TM}$ (Chapter 14).

## 2. Definitions

### The hierarchy as data

In [ ]:
HIER = [
 ("DFA/NFA", "regular",             "right-linear",      "(01)*",        "Ch 4-7"),
 ("DPDA",    "deterministic CFL",   "LR(k)",             "w # w^R",      "Ch 11, 12"),
 ("NPDA",    "context-free",        "context-free",      "w w^R, Dyck",  "Ch 11, 12"),
 ("LBA",     "context-sensitive",   "context-sensitive", "a^n b^n c^n",  "Ch 13"),
 ("TM",      "recursively enum.",   "unrestricted",      "A_TM",         "Ch 13, 14"),
]

SEPARATORS = [
 ("regular  <  DCFL", "a^n b^n",     "Chapter 4 pumping lemma"),
 ("DCFL     <  CFL",  "w w^R",       "Chapter 11, Concept 17"),
 ("CFL      <  CSL",  "a^n b^n c^n", "Chapter 11, Concept 19"),
 ("CSL      <  RE",   "A_TM",        "Chapter 14"),
]

# --- thin wrappers over Jove's TM runner --------------------------------
# run_tm(T, tape, fuel) returns (truncated-paths, haltList).  A TM HALTS
# when no transition applies, and ACCEPTS if it halts in a final state.
# So an accepting state must have NO outgoing transitions, or the machine
# will run on past it.
def tm_accepts(T, tape, fuel=200):
    trunc, halts = run_tm(T, tape if tape != '' else '.', fuel, chatty=False)
    return any(cfg[0] in T["F"] for cfg, _ in halts)

def tm_halts(T, tape, fuel=200):
    trunc, halts = run_tm(T, tape if tape != '' else '.', fuel, chatty=False)
    return len(halts) > 0

def tm_tape(T, tape, fuel=200):
    trunc, halts = run_tm(T, tape if tape != '' else '.', fuel, chatty=False)
    return [cfg[2].rstrip('.') for cfg, _ in halts]

## 3. Tests

The hierarchy.

In [ ]:
print("%-10s %-22s %-20s %-14s %s"
      % ("machine", "languages", "grammars", "example", "where"))
for row in HIER:
    print("%-10s %-22s %-20s %-14s %s" % row)

Every containment is **strict**, with a witness.

In [ ]:
for rel, wit, where in SEPARATORS:
    print("  %-20s witness %-14s (%s)" % (rel, wit, where))
assert len(SEPARATORS) == 4

Witness 1: $a^nb^n$ separates regular from context-free.

In [ ]:
def in_anbn(s):
    k = len(s) - len(s.lstrip('a'))
    return s == 'a'*k + 'b'*(len(s)-k) and k == len(s)-k
for N in range(2, 6):
    w = 'a'*N + 'b'*N
    splits = [(w[:i], w[i:j], w[j:]) for i in range(N+1)
              for j in range(i+1, min(N, len(w)) + 1)]
    assert all(any(not in_anbn(x + y*k + z) for k in range(3)) for x, y, z in splits)
print("every pumping split of a^N b^N breaks -> not regular")
print("and the PDA of Chapter 12 recognises it -> context-free")

Witness 3: $a^nb^nc^n$ separates context-free from context-sensitive.

In [ ]:
def in_abc(s):
    i = len(s) - len(s.lstrip('a')); rest = s[i:]
    j = len(rest) - len(rest.lstrip('b')); k = len(rest) - j
    return s == 'a'*i + 'b'*j + 'c'*k and i == j == k
def splits5(w, N):
    out, n = [], len(w)
    for i in range(n+1):
        for j in range(i, n+1):
            for k in range(j, n+1):
                for l in range(k, n+1):
                    u, v, x, y, z = w[:i], w[i:j], w[j:k], w[k:l], w[l:]
                    if (v or y) and len(v+x+y) <= N: out.append((u,v,x,y,z))
    return out
for N in [2, 3]:
    w = 'a'*N + 'b'*N + 'c'*N
    sp = splits5(w, N)
    assert all(any(not in_abc(u + v*i + x + y*i + z) for i in range(3))
               for (u, v, x, y, z) in sp)
print("every CFL pumping split of a^N b^N c^N breaks -> not context-free")
print("a linear bounded automaton recognises it    -> context-sensitive")

A TM can recognise it, which places it inside RE as well.

In [ ]:
Abc = md2mc('''TM
!! cross off one a, one b, one c per sweep
I  : X ; X , R -> I
I  : a ; X , R -> Sb
I  : Y ; Y , R -> I
I  : Z ; Z , R -> D
I  : . ; . , S -> F
Sb : a ; a , R -> Sb
Sb : Y ; Y , R -> Sb
Sb : b ; Y , R -> Sc
Sc : b ; b , R -> Sc
Sc : Z ; Z , R -> Sc
Sc : c ; Z , L -> Bk
Bk : a ; a , L -> Bk
Bk : b ; b , L -> Bk
Bk : c ; c , L -> Bk
Bk : X ; X , L -> Bk
Bk : Y ; Y , L -> Bk
Bk : Z ; Z , L -> Bk
''')
print("a TM for a^n b^n c^n has %d states" % len(Abc["Q"]))
print("\nThe point is not this machine's details -- it is that ONE tape with")
print("write access already reaches beyond context-free.")

Reading the table the other way: each row adds exactly one capability.

In [ ]:
ADDS = [("DFA -> PDA", "a stack: unbounded LIFO memory"),
        ("PDA -> LBA", "a rewritable tape, but only as long as the input"),
        ("LBA -> TM",  "an unbounded tape")]
for a, b in ADDS: print("  %-14s %s" % (a, b))

## 4. Exercises


1. Where do the **deterministic** CFLs sit, and why is that row special?
2. Give a language in CSL but not CFL other than $a^nb^nc^n$.
3. Is every context-sensitive language decidable? Is every RE language?

In [ ]:
# Your work for the exercises above.